In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.testing_backfill_aws_drop_part1_20251020;
CREATE TABLE dev.mohit_gangwani.testing_backfill_aws_drop_part1_20251020 AS
WITH null_gaps AS (
  SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
      , LEAD(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
      , LEAD(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_end
      , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
      , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
      , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
      , LEAD(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_station_id
      , LAG(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_station_id
      , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
      , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
      , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
      , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
      , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
      , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
      , LEAD(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_airdate
      , LAG(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_airdate
      , LEAD(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_airdate
      , LAG(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_airdate
      , LEAD(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_station_id
      , LAG(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_station_id
      , LEAD(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_show_id
      , LAG(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_show_id
      , LEAD(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_fk_content_id
      , LAG(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_fk_content_id
      , TIMESTAMPADD(SECOND, prev_runtime + 120, prev_airdate) AS prev_airdate_end
      , TIMESTAMPADD(SECOND, next_runtime + 120, next_airdate) AS next_airdate_end
      , CASE WHEN prev_end <= prev_airdate_end THEN TRUE ELSE FALSE END AS prev_is_live
      , CASE WHEN next_end <= next_airdate_end THEN TRUE ELSE FALSE END AS next_is_live
      FROM prod.detection.viewing_content_firehose
      WHERE session_start >= '2025-10-22 00:00:00'
        AND session_start < '2025-10-22 01:00:00'
),
null_calc AS (
    SELECT *, 
            CASE 
                WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id THEN 1
                WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL THEN 2
                WHEN next_show_id IS NOT NULL AND next_show_id IS NULL THEN 3
                WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
                    THEN CASE WHEN NVL(next_media_time_start, 0) - session_duration >= 0
                                AND next_input_source_id <=> fk_input_source_id
                                AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                                AND NVL(next_is_live, FALSE) THEN 3
                                WHEN NVL(prev_media_time_end, 0) + session_duration <= NVL(prev_runtime, 0)
                                AND prev_input_source_id <=> fk_input_source_id
                                AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                                AND NVL(prev_is_live, FALSE) THEN 2
                        END
            END AS num_for_calc
        FROM null_gaps
        WHERE fk_show_id IS NULL
        AND session_duration <= 180
        AND next_start >= session_end
        AND prev_end <= session_start
        AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
            THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
            AND prev_media_time_end + session_duration <= next_media_time_start
            AND next_is_live
            AND prev_is_live
            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
            THEN prev_input_source_id <=> fk_input_source_id
            AND prev_media_time_end + session_duration <= prev_runtime
            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
            AND prev_is_live
            WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
            THEN next_input_source_id <=> fk_input_source_id
            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
            AND next_media_time_start - session_duration >= 0
            AND next_is_live
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
            THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                        THEN prev_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                            AND prev_is_live
                        WHEN next_media_time_start - session_duration >= 0
                        THEN next_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                        AND next_is_live
                        ELSE FALSE END
            ELSE FALSE END
    GROUP BY ALL

),
null_gaps_to_fill AS (
    SELECT 
        fk_tvid
        , session_start
        , session_end
        , session_duration
        , fk_input_source_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end
            WHEN num_for_calc = 3 THEN next_media_time_start - session_duration
        END AS media_time_start
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end + session_duration
            WHEN num_for_calc = 3 THEN next_media_time_start
        END AS media_time_end
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_station_id
            WHEN num_for_calc = 3 THEN next_station_id
        END AS fk_station_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_show_id
            WHEN num_for_calc = 3 THEN next_show_id
        END AS fk_show_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_airdate
            WHEN num_for_calc = 3 THEN next_airdate
        END AS airdate
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_runtime
            WHEN num_for_calc = 3 THEN next_runtime
        END AS runtime
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_airdate
            WHEN num_for_calc = 3 THEN next_tms_airdate
        END AS tms_airdate
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_station_id
            WHEN num_for_calc = 3 THEN next_tms_station_id
        END AS tms_station_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_show_id
            WHEN num_for_calc = 3 THEN next_tms_show_id
        END AS tms_show_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_fk_content_id
            WHEN num_for_calc = 3 THEN next_fk_content_id
        END AS fk_content_id
        , TRUE AS is_live
        , fk_input_source_id
    FROM null_calc
    GROUP BY ALL
),
comms AS (
    SELECT 
        vc.fk_tvid, 
        vc.session_start, 
        vc.session_end
    FROM detection.viewing_commercials_firehose AS vc
    JOIN null_gaps_to_fill AS tv
        ON tv.fk_tvid = vc.fk_tvid
        AND vc.session_start <= tv.session_end
        AND vc.session_end >= tv.session_start
    WHERE partition_key >= DATE('2025-10-22 00:00:00')
        AND partition_key <= DATE('2025-10-22 01:00:00')
    GROUP BY ALL
    ORDER BY 1, 2, 3
),
comm_duration AS (
    SELECT 
        fk_tvid, 
        tv_group_num
        , MIN(session_start) AS session_start
        , MAX(session_end) AS session_end
        , TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
    FROM (
    SELECT 
        *, 
        SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
        SELECT fk_tvid, session_start, session_end
        , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
        , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
        FROM comms
    )
    )
    GROUP BY 1, 2
),
filled_null_gaps AS( 
    SELECT 
        fk_tvid, session_start, session_end, session_duration, fk_input_source_id
    , media_time_start, media_time_end, is_live
    , runtime, airdate, fk_show_id, fk_station_id, tms_airdate, tms_show_id, tms_station_id, fk_content_id
    , SUM(comm_duration)*1.0 AS total_comm_duration
    FROM (
    SELECT content.fk_tvid, content.session_start, content.session_end, content.session_duration, content.fk_input_source_id
    , content.media_time_start, content.media_time_end, content.is_live, content.fk_show_id, content.airdate
    , content.runtime, content.fk_input_source_id
    , content.fk_station_id, content.tms_airdate, content.tms_show_id, content.tms_station_id, content.fk_content_id
    , GREATEST(comms.session_start, content.session_start) AS comm_start
    , LEAST(comms.session_end, content.session_end) AS comm_end
    , TIMESTAMPDIFF(SECOND, comm_start, comm_end) AS comm_duration
    FROM null_gaps_to_fill AS content
    JOIN comm_duration comms
        ON content.fk_tvid = comms.fk_tvid
        AND comms.session_start <= content.session_end
        AND comms.session_end >= content.session_start
    WHERE content.fk_show_id IS NOT NULL
        AND content.airdate IS NOT NULL
        AND content.media_time_start IS NOT NULL
        AND content.fk_station_id IS NOT NULL
    )
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16
),
filled_null_gaps_filter AS (
    SELECT 
        *
    FROM filled_null_gaps
    WHERE 
        CASE WHEN session_duration <= 15 
            THEN total_comm_duration >= 5
        ELSE total_comm_duration/session_duration >= 0.5 
    END
)
SELECT 
    c.fk_tvid
    , c.session_start
    , c.session_end
    , COALESCE(n.session_duration, c.session_duration) AS session_duration
    , COALESCE(n.fk_show_id, c.fk_show_id) AS fk_show_id
    , COALESCE(n.fk_station_id, c.fk_station_id) AS fk_station_id
    , COALESCE(n.airdate, c.airdate) AS airdate
    , COALESCE(n.tms_airdate, c.tms_airdate) AS tms_airdate
    , COALESCE(n.tms_show_id, c.tms_show_id) AS tms_show_id
    , COALESCE(n.tms_station_id, c.tms_station_id) AS tms_station_id
    , COALESCE(n.media_time_start, c.media_time_start) AS media_time_start
    , COALESCE(n.runtime, c.runtime) AS runtime
    , COALESCE(n.fk_content_id, c.fk_content_id) AS fk_content_id
    , c.fk_input_source_id
    , c.fk_dma_id
    , c.fk_location_id
    , COALESCE(n.is_live, c.is_live) AS is_live
    , c.fk_zoo_id
    , c.tms_tuner_channel_id
    , c.tms_tuner_program_id
    , c.tuner_channel_number
    , c.tuner_channel_id
    , c.tuner_program_id
    , c.vizio_epg_program
    , c.vizio_epg_station
    , c.reported_input_source
    , c.content_type
    , c.file_ingested
    , CASE WHEN n.fk_tvid IS NOT NULL THEN 1 
    END AS if_filled
FROM prod.detection.viewing_content_firehose c
LEFT JOIN filled_null_gaps_filter n
ON n.fk_tvid = c.fk_tvid
AND n.session_start = c.session_start
AND n.session_end = c.session_end
AND n.fk_input_source_id = c.fk_input_source_id
WHERE c.session_start >= '2025-10-22 00:00:00'
  AND c.session_start < '2025-10-22 01:00:00'
ORDER BY 1, 2, 3;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.testing_backfill_aws_drop_golden_20251020;
CREATE TABLE dev.mohit_gangwani.testing_backfill_aws_drop_golden_20251020 AS
WITH activity_obfuscation AS (
  SELECT blocked_apps.app_name, override.client_name
  FROM prod.detection.app_activity_distribution_blacklist AS blocked_apps
  LEFT JOIN detection.app_customer_activity_distribution_override override
    ON blocked_apps.app_name = override.app_name
  GROUP BY 1, 2
)
, viewing_obfuscation AS (
  SELECT blocked_apps.app_name, override.client_name
  FROM prod.detection.app_viewing_distribution_blacklist AS blocked_apps
  LEFT JOIN detection.app_customer_viewing_distribution_override override
    ON blocked_apps.app_name = override.app_name
  GROUP BY 1, 2
)
, vod_stations AS (
  SELECT station_id, vendor_name
  FROM prod.detection.station_distribution_obfuscation_overwrite
  GROUP BY 1, 2
  )
, national AS (
  SELECT rl.station_id, 
  rl.fk_show_id, 
  rl.tuner_channel_id, 
  rl.tuner_program_id, 
  rl.airdate
  FROM prod.detection.nielsen_replacement_national_nyc AS rl
  JOIN prod.detection.nielsen_only_distribution_blacklist AS bl
    ON bl.station_id = rl.station_id
   AND bl.blacklist_end >= '2025-10-22 00:00:00'::timestamp
  GROUP BY ALL
)
, local AS (
  SELECT rl.station_id, 
  rl.fk_show_id, 
  rl.dma_id, 
  rl.tuner_channel_id, 
  rl.tuner_program_id, 
  rl.airdate
  FROM prod.detection.nielsen_replacement_local AS rl
  JOIN prod.detection.nielsen_only_distribution_blacklist AS bl
    ON bl.station_id = rl.station_id
   AND bl.blacklist_end >= '2025-10-22 00:00:00'::timestamp
  GROUP BY ALL
)
, inscape_map_deduped AS (
  SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id, channel_affiliate
  FROM (
    SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id
    , CASE WHEN st.inscape_station_name IS NOT NULL THEN st.inscape_station_name
    WHEN LOWER(st.station_affil) LIKE '%affiliate%' OR LOWER(st.station_affil) LIKE '%independent%' OR LOWER(st.station_affil) LIKE '%low power%' THEN st.station_affil END AS channel_affiliate
    , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
    FROM prod.detection.inscape_station_map ism
    JOIN prod.detection.epg_station st
      ON st.station_id = ism.mapped_vendor_station_id
     AND st.vendor_name = ism.mapped_vendor
  ) ism
  WHERE ism.rn = 1
)
SELECT tvid
, fk_tvid
, zipcode
, dma
, tms_episode_id
, tivo_episode_id
, tms_title
, tivo_title
, tms_airdate
, tivo_airdate
, tms_channel_callsign
, tivo_channel_callsign
, mt_start
, session_start
, session_end
, tms_channel_affiliate
, tivo_channel_affiliate
, is_live
, ip_address
, reported_input_source
, content_type
, tuner_content_type
, input_category
, input_device
, app_service
, tuner_tms_episode_id
, tuner_tivo_episode_id
, tuner_tms_title
, tuner_tivo_title
, tuner_tms_airdate
, tuner_tivo_airdate
, tuner_tms_channel_callsign
, tuner_tivo_channel_callsign
, tuner_mt_start
, tuner_tms_channel_affiliate
, tuner_tivo_channel_affiliate
, tuner_is_live
, tuner_input_category
, tuner_input_device
, tuner_app_service
, tuner_channel_number
, enableaudioacr
, dma_code
, vizio_epg_channel_id
, vizio_epg_program_id
, tms_show_genre
, tivo_show_genre
, tms_epi_title
, tivo_epi_title
, series_id
, show_duration
, vizio_epg_not_null
, nielsen_exclusive
, content_only_condition
, tuner_content_only_condition
, vod_station
, '|'||array_join(collect_set(acrb_client), '|')||'|' AS acrb_clients
, '|'||array_join(collect_set(appb_client), '|')||'|' AS appb_clients
, '|'||array_join(collect_set(client_id), '|')||'|' AS client_id_not_null
, DATE_TRUNC('HOUR', session_start) AS session_start_hour
FROM (
  SELECT DISTINCT COALESCE(tv.long_tvid, tv.vizio_tvid) AS tvid
  , c.fk_tvid
  , NULLIF(location.zipcode, '') AS zipcode
  , REPLACE(dma.dma_name, ',', '') AS dma
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN
          CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL 
              ELSE vizio_program.program_tms_id END
          WHEN c.file_ingested = true THEN COALESCE(md.external_id,SPLIT(cid.content_cid, '_')[0])
          ELSE tms_show.database_key
      END AS tms_episode_id
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN NULL
          WHEN c.file_ingested = true THEN COALESCE(md.external_id,SPLIT(cid.content_cid, '_')[0])
          ELSE tivo_show.database_key
      END AS tivo_episode_id
  , REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN
          CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL 
              WHEN vizio_program.series_aggregate_title IS NOT NULL AND vizio_program.series_aggregate_title != '' THEN vizio_program.series_aggregate_title
              ELSE vizio_program.title END
          WHEN c.file_ingested THEN NULL
          ELSE tms_show.title
      END, '[\“\”\"\^\@\,]', '') AS tms_title
  , REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN
          CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL 
              WHEN vizio_program.series_aggregate_title IS NOT NULL AND vizio_program.series_aggregate_title != '' THEN vizio_program.series_aggregate_title
              ELSE vizio_program.title END
          WHEN c.file_ingested THEN NULL
          ELSE tivo_show.title
      END, '[\“\”\"\^\@\,]', '') AS tivo_title
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL
          WHEN c.vizio_epg_station IS NOT NULL THEN c.tms_airdate
          WHEN COALESCE(cid.content_cid, 'x') = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
          ELSE c.tms_airdate
      END AS tms_airdate
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL
          WHEN c.vizio_epg_station IS NOT NULL THEN c.airdate
          WHEN COALESCE(cid.content_cid, 'x') = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
          ELSE c.airdate
      END AS tivo_airdate
  , CASE WHEN tms_station_obfs.station_id IS NOT NULL THEN NULL
          WHEN c.vizio_epg_station IS NOT NULL THEN tms_map.inscape_call_sign
          WHEN c.file_ingested = true THEN SPLIT(cid.content_cid, '_')[1]
          ELSE tms_map.inscape_call_sign
      END AS tms_channel_callsign
  , CASE WHEN tivo_station_obfs.station_id IS NOT NULL THEN NULL
          WHEN c.vizio_epg_station IS NOT NULL THEN tivo_map.inscape_call_sign
          WHEN c.file_ingested = true THEN SPLIT(cid.content_cid, '_')[1]
          ELSE tivo_map.inscape_call_sign
      END AS tivo_channel_callsign
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL 
          WHEN c.vizio_epg_station IS NOT NULL THEN c.media_time_start
          WHEN COALESCE(cid.content_cid, 'x') = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
          ELSE LEAST(c.media_time_start, c.runtime)
      END AS mt_start
  , c.session_start
  , c.session_end
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN
              CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN 'OBFUSCATED' 
                  ELSE vizio_station.name END
          WHEN tms_station_obfs.station_id IS NOT NULL THEN NULL
          ELSE tms_map.channel_affiliate
      END AS tms_channel_affiliate
  
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN
              CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN 'OBFUSCATED' 
                  ELSE vizio_station.name END
          WHEN tivo_station_obfs.station_id IS NOT NULL THEN NULL
          ELSE tivo_map.channel_affiliate
      END AS tivo_channel_affiliate
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN 't'
          WHEN COALESCE(cid.content_cid, 'x') = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
          WHEN c.is_live = TRUE THEN 't'
          WHEN c.is_live = FALSE THEN 'f'
      END AS is_live
  , ip.ip_address
  , c.reported_input_source
  , CASE WHEN c.reported_input_source = 'APPS' THEN 'APPS'
          ELSE tvis.category 
  END AS input_category
  , CASE 
      WHEN c.reported_input_source = 'APPS' THEN NULL
      WHEN c.reported_input_source = 'ANTENNA' AND tvis.input_device = 'OTA' THEN tvis.input_device
      WHEN c.reported_input_source = 'ANTENNA' AND tvis.input_device != 'OTA' THEN NULL
      ELSE tvis.input_device
  END AS input_device -- changes to input_device
  , CASE WHEN UPPER(tvis.category) = 'APPS' THEN
          CASE WHEN c.vizio_epg_station IS NOT NULL THEN 'WatchFree+'
              WHEN c.vizio_epg_station IS NULL and tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
              WHEN LOWER(tis.app_name) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora',  'tv games')
                  AND COALESCE(cid.content_cid, 'x') <> 'unknown' THEN NULL
              WHEN lower(tis.app_name) = 'unknown' THEN NULL
              ELSE tis.app_name
          END
      WHEN UPPER(c.reported_input_source) = 'APPS' 
          AND c.is_live = true 
          AND (LOWER(tvis.input_device) IN ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4', 'playstation 5','roku') OR tvis.input_device IS NULL )THEN NULL
          WHEN UPPER(c.reported_input_source) = 'ANTENNA' AND c.is_live = true 
          AND LOWER(tvis.input_device) IN ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4', 'playstation 5','roku') THEN NULL --changes
          WHEN c.is_live = true AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4',  'playstation 5','roku') THEN 'vMVPD'
    END AS app_service
  , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.database_key END  AS tuner_tms_episode_id
  , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.database_key END AS tuner_tivo_episode_id
  , REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.title END, '[\“\”\"\^\@\,]', '') AS tuner_tms_title
  , REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.title END, '[\“\”\"\^\@\,]', '') AS tuner_tivo_title
  , CASE WHEN c.file_ingested THEN NULL ELSE c.tms_airdate END AS tuner_tms_airdate
  , CASE WHEN c.file_ingested THEN NULL ELSE c.airdate     END AS tuner_tivo_airdate
  , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_map.inscape_call_sign  END AS tuner_tms_channel_callsign
  , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_map.inscape_call_sign END AS tuner_tivo_channel_callsign
  , CASE WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL
              THEN LEAST((unix_timestamp(c.session_start)-unix_timestamp(COALESCE(c.airdate, c.tms_airdate))), c.runtime) 
          ELSE LEAST(c.media_time_start, c.runtime)
      END AS tuner_mt_start
  , tuner_tms_map.channel_affiliate  AS tuner_tms_channel_affiliate
  , tuner_tivo_map.channel_affiliate AS tuner_tivo_channel_affiliate
  , CASE WHEN COALESCE(c.tuner_channel_id, NULLIF(c.tms_tuner_channel_id,98989898)) IS NOT NULL THEN 't'
          WHEN c.is_live = TRUE THEN 't'
          WHEN c.is_live = FALSE THEN 'f'
      END AS tuner_is_live
  , CASE 
      WHEN c.reported_input_source = 'APPS' THEN 'APPS'
      WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL AND UPPER(tvis.category) in ('APPS', 'OTHER', 'OTT') THEN 'HD TV'
      WHEN UPPER(tvis.category) = 'OTHER' AND tis.app_name = 'WatchFree+' AND COALESCE(cid.content_cid, 'x') != 'unknown' THEN 'HD TV' 
      WHEN UPPER(tvis.category) = 'OTT' AND tis.app_name = 'WatchFree+' THEN 'APPS'
      ELSE tvis.category
  END AS tuner_input_category
  , CASE  
      WHEN c.reported_input_source = 'APPS' THEN NULL
      WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN 'OTA'
      WHEN inps.input_source = 'DTV' THEN 'OTA'
      WHEN c.reported_input_source = 'ANTENNA' AND tvis.input_device = 'OTA' THEN tvis.input_device
      WHEN c.reported_input_source = 'ANTENNA' AND tvis.input_device != 'OTA' THEN NULL
      ELSE tvis.input_device 
  END AS tuner_input_device
  , CASE 
      WHEN UPPER(C.reported_input_source) = 'APPS' AND c.is_live = true 
      AND (LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4',  'playstation 5','roku') OR tvis.input_device IS NULL) THEN NULL
      WHEN UPPER(C.reported_input_source) = 'ANTENNA' AND c.is_live = true 
      AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4',  'playstation 5','roku') THEN NULL
      WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL 
      AND NVL(tvis.input_device, 'OTA') = 'OTA' THEN 'WatchFree+'
      WHEN UPPER(tvis.category) = 'APPS' THEN
          CASE 
              WHEN c.vizio_epg_station IS NOT NULL THEN 'WatchFree+'
              WHEN c.vizio_epg_station IS NULL AND tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
              WHEN LOWER(tis.app_name) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora','tv games') AND COALESCE(cid.content_cid, 'x') <> 'unknown' THEN NULL
              WHEN lower(tis.app_name) = 'unknown' THEN NULL
              ELSE tis.app_name
          END
      WHEN c.is_live = true AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4','playstation 5','roku')
      AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NULL THEN 'vMVPD'
      WHEN inps.input_source IN ('DTV', 'TUNER', 'COAXIAL', 'ATV')
      AND NVL(tvis.input_device, 'OTA') = 'OTA'
      AND tis.app_name ='WatchFree+'
      AND c.is_live = TRUE THEN 'WatchFree+'
  END AS tuner_app_service
  , c.tuner_channel_number
  , CASE WHEN settings.enableaudioacr = 1 THEN 't' ELSE 'f' END AS enableaudioacr
  , dma.dma_code AS dma_code
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL
          ELSE vizio_station.station_id
      END AS vizio_epg_channel_id
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL
          ELSE vizio_program.program_aggregate_id
      END AS vizio_epg_program_id
  , NULLIF(REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN REGEXP_REPLACE(vizio_program.aggregate_genres, '[\\[|\\]]', '')
                  ELSE tms_show.genre
          END, ', ?', '|'), '') AS tms_show_genre
  , NULLIF(REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN REGEXP_REPLACE(vizio_program.aggregate_genres, '[\\[|\\]]', '')
                  ELSE tivo_show.genre
          END, ', ?', '|'), '') AS tivo_show_genre
  , NULLIF(REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN vizio_program.title
                  ELSE tms_show.epi_title END, '[\“\”\"\^\@\,]', ''), ''
      ) AS tms_epi_title
  , NULLIF(REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN vizio_program.title
                  ELSE tivo_show.epi_title END, '[\“\”\"\^\@\,]', ''), ''
      ) AS tivo_epi_title
  , CASE WHEN c.file_ingested THEN NULL ELSE tivo_show.series_id END AS series_id
  , CASE WHEN c.file_ingested THEN NULL ELSE c.runtime END AS show_duration
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN TRUE ELSE FALSE END AS vizio_epg_not_null
  , CASE WHEN COALESCE(tivo_nielsen_blacklist.station_id, tms_nielsen_blacklist.station_id) IS NOT NULL
          AND (COALESCE(tivo_rep_local.station_id, tivo_rep_nyc_nat.station_id, tms_rep_local.station_id, tms_rep_nyc_nat.station_id) IS NULL
              OR COALESCE(tms_nielsen_blacklist.ingest_time, tivo_nielsen_blacklist.ingest_time) IS NOT NULL) THEN TRUE
          ELSE FALSE
      END AS nielsen_exclusive
  , CASE WHEN COALESCE(cid.content_cid, 'x') = 'unknown' AND vizio_station.name IS NULL THEN TRUE ELSE FALSE END AS content_only_condition
  , CASE WHEN COALESCE(cid.content_cid, 'x') = 'unknown'
          AND vizio_station.name IS NULL
          AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NULL THEN TRUE
          ELSE FALSE
      END AS tuner_content_only_condition
  ,  CASE WHEN c.reported_input_source = 'ANTENNA' THEN FALSE 
          WHEN COALESCE(tivo_vod_stations.station_id, tms_vod_stations.station_id) IS NOT NULL THEN TRUE 
          ELSE FALSE
    END AS vod_station
  , CASE WHEN acrb.app_name IS NOT NULL AND c.vizio_epg_station IS NULL THEN
        CASE WHEN acrb.client_name IS NULL THEN 'ALL' ELSE acrb.client_name
        END
    END AS acrb_client
  , CASE WHEN appb.app_name IS NOT NULL AND c.vizio_epg_station IS NULL THEN
        CASE WHEN appb.client_name IS NULL THEN 'ALL' ELSE appb.client_name
        END
    END AS appb_client
  , cl.client_name AS client_id
  , c.content_type 
  , CASE WHEN c.tuner_channel_number IS NOT NULL then 'LINEAR' ELSE c.content_type END AS tuner_content_type 
  FROM dev.mohit_gangwani.testing_backfill_aws_drop_part1_20251020 AS c
  JOIN prod.detection.zoo AS z
      ON c.fk_zoo_id = z.zoo_id
  AND z.zoo = 'control-zoo-dtsprod.tvinteractive.tv'
  JOIN prod.detection.tv AS tv
      ON c.fk_tvid = tv.tvid
  AND tv.oem = 'VIZIO'
  JOIN prod.detection.tv_settings AS tv_settings
      ON c.session_start >= tv_settings.create_timestamp
  AND c.session_start < tv_settings.next_create_timestamp
  AND tv_settings.create_timestamp <= '2025-10-22 01:00:00'::timestamp
  AND tv_settings.next_create_timestamp >= '2025-10-22 00:00:00'::timestamp
  AND c.fk_tvid = tv_settings.fk_tvid
  JOIN prod.detection.settings AS settings
      ON tv_settings.fk_settings_id = settings.settings_id
  AND UPPER(settings.country_name) = 'USA'
  JOIN prod.detection.tv_populations AS u
      ON c.fk_tvid = u.fk_tvid
  JOIN prod.detection.populations AS pop
      ON u.fk_population_id = pop.population_id
  AND pop.population_name = 'opted_in'
  JOIN prod.detection.location AS location
      ON c.fk_location_id = location.location_id
  AND UPPER(location.country_code) = 'US'
  LEFT OUTER JOIN prod.detection.dma AS dma
      ON c.fk_dma_id = dma.dma_id
  LEFT JOIN prod.detection.content_ids_firehose AS cid
      ON cid.content_id = c.fk_content_id
  LEFT OUTER JOIN prod.detection.tv_ip_address AS ip
      ON c.session_start >= ip.create_timestamp
  AND c.session_start < ip.next_create_timestamp
  AND ip.create_timestamp <= '2025-10-22 01:00:00'::timestamp
  AND ip.next_create_timestamp >= '2025-10-22 00:00:00'::timestamp
  AND tv.tvid = ip.fk_tvid
  LEFT OUTER JOIN prod.detection.vizio_epg_station AS vizio_station
      ON TRY_CAST(c.vizio_epg_station AS STRING) = TRY_CAST(vizio_station.station_id AS STRING)
  LEFT OUTER JOIN prod.detection.vizio_epg_program_aggregate AS vizio_program
      ON TRY_CAST(c.vizio_epg_program AS STRING) = TRY_CAST(vizio_program.program_aggregate_id AS STRING)
  AND TRY_CAST(c.vizio_epg_program AS STRING) NOT IN ('0', '', '-1')
  LEFT OUTER JOIN prod.detection.free_channels_distribution_blacklist AS chanb
      ON vizio_station.name = chanb.channel_name
  LEFT OUTER JOIN prod.detection.input_source AS inps
      ON c.fk_input_source_id = inps.input_source_id
  JOIN prod.detection.tv_input_stats_firehose AS tvis
      ON c.session_start >= tvis.create_timestamp
  AND c.session_start < tvis.next_create_timestamp
  AND tvis.create_timestamp <= '2025-10-22 01:00:00'::timestamp
  AND tvis.next_create_timestamp >= '2025-10-22 00:00:00'::timestamp
  AND  c.fk_tvid = tvis.fk_tvid
  AND  c.fk_input_source_id = tvis.fk_input_source_id
  LEFT OUTER JOIN prod.detection.tv_inputsource AS tis
      ON c.session_start >= (tis.create_timestamp::double)::timestamp
  AND c.session_start < (tis.next_create_timestamp::double)::timestamp
  AND c.fk_tvid = tis.fk_tvid
  AND c.fk_input_source_id = tis.fk_input_source_id
  AND tis.create_timestamp <= '2025-10-22 01:00:00'::timestamp
  AND tis.next_create_timestamp >= '2025-10-22 00:00:00'::timestamp
  LEFT OUTER JOIN activity_obfuscation AS appb
      ON tis.app_name = appb.app_name
  LEFT OUTER JOIN viewing_obfuscation AS acrb
      ON tis.app_name = acrb.app_name
  LEFT OUTER JOIN prod.detection.content_id_external_firehose AS m
      ON m.fk_content_id = c.fk_content_id
  LEFT OUTER JOIN prod.detection.clients AS cl
      ON m.fk_client_id = cl.client_id
  LEFT OUTER JOIN prod.detection.content_id_external_firehose AS md
      ON md.fk_content_id = c.fk_content_id
  LEFT OUTER JOIN prod.detection.clients AS cli
      ON md.fk_client_id = cli.client_id
  LEFT OUTER JOIN inscape_map_deduped AS tivo_map
      ON tivo_map.mapped_vendor_station_id = c.fk_station_id
  AND tivo_map.mapped_vendor = 'TIVO'
  LEFT OUTER JOIN prod.detection.epg_show AS tivo_show
      ON tivo_show.show_id = c.fk_show_id
  AND tivo_show.vendor_name = 'TIVO'
  LEFT OUTER JOIN prod.detection.epg_show AS tms_show
      ON tms_show.show_id = c.tms_show_id
  AND tms_show.vendor_name = 'TMS'
  LEFT OUTER JOIN inscape_map_deduped AS tms_map
      ON tms_map.mapped_vendor_station_id = c.tms_station_id
  AND tms_map.mapped_vendor = 'TMS'
  LEFT OUTER JOIN inscape_map_deduped AS tuner_tivo_map
      ON tuner_tivo_map.mapped_vendor_station_id = c.tuner_channel_id
  AND tuner_tivo_map.mapped_vendor = 'TIVO'
  LEFT OUTER JOIN detection.epg_show AS tuner_tivo_show
      ON tuner_tivo_show.show_id = c.tuner_program_id
  AND tuner_tivo_show.vendor_name = 'TIVO'
  LEFT OUTER JOIN prod.detection.epg_show AS tuner_tms_show
      ON tuner_tms_show.show_id = c.tms_tuner_program_id
  AND tuner_tms_show.vendor_name = 'TMS'
  LEFT OUTER JOIN inscape_map_deduped AS tuner_tms_map
      ON tuner_tms_map.mapped_vendor_station_id = c.tms_tuner_channel_id
  AND tuner_tms_map.mapped_vendor = 'TMS'
  LEFT OUTER JOIN vod_stations AS tivo_vod_stations
      ON tivo_vod_stations.station_id = tivo_map.inscape_station_id
  AND tivo_vod_stations.vendor_name = 'TIVO'
  LEFT OUTER JOIN vod_stations AS tms_vod_stations
      ON tms_vod_stations.station_id = tms_map.inscape_station_id
  AND tms_vod_stations.vendor_name = 'TMS'
  LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS tivo_station_obfs
      ON tivo_station_obfs.vendor_station_id = tivo_map.inscape_station_id
  LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS tms_station_obfs
      ON tms_station_obfs.vendor_station_id = tms_map.inscape_station_id
  LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS tivo_nielsen_blacklist
      ON tivo_nielsen_blacklist.station_id = tivo_map.inscape_station_id
  AND c.session_start >= tivo_nielsen_blacklist.blacklist_start
  AND c.session_start < tivo_nielsen_blacklist.blacklist_end
  LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS tms_nielsen_blacklist
      ON tms_nielsen_blacklist.station_id = tms_map.inscape_station_id
  AND c.session_start >= tms_nielsen_blacklist.blacklist_start
  AND c.session_start < tms_nielsen_blacklist.blacklist_end
  LEFT OUTER JOIN local AS tivo_rep_local
      ON tivo_rep_local.station_id = tivo_map.inscape_station_id
  AND tivo_rep_local.airdate = c.airdate
  AND tivo_rep_local.fk_show_id = c.fk_show_id
  AND tivo_rep_local.dma_id = c.fk_dma_id
  LEFT OUTER JOIN local AS tms_rep_local
      ON tms_rep_local.station_id = tms_map.inscape_station_id
  AND tms_rep_local.airdate = c.tms_airdate
  AND tms_rep_local.fk_show_id = c.tms_show_id
  AND tms_rep_local.dma_id = c.fk_dma_id
  LEFT OUTER JOIN national AS tivo_rep_nyc_nat
      ON tivo_rep_nyc_nat.station_id = tivo_map.inscape_station_id
  AND tivo_rep_nyc_nat.airdate = c.airdate
  AND tivo_rep_nyc_nat.fk_show_id = c.fk_show_id
  LEFT OUTER JOIN national AS tms_rep_nyc_nat
      ON tms_rep_nyc_nat.station_id = tms_map.inscape_station_id
  AND tms_rep_nyc_nat.airdate = c.tms_airdate
  AND tms_rep_nyc_nat.fk_show_id = c.tms_show_id
  WHERE CASE c.file_ingested
      WHEN true THEN
          CASE NULLIF(SPLIT(cid.content_cid, '_')[1], '') IS NOT NULL AND NULLIF(SPLIT_PART(cid.content_cid, '_', 3), '') IS NULL
          WHEN true THEN SPLIT(cid.content_cid, '_')[1]
          ELSE NULL
          END
      ELSE COALESCE(tivo_map.inscape_call_sign, tms_map.inscape_call_sign, 'KeepSessionForNullReport')
      END NOT IN (SELECT DISTINCT chan_callsign FROM prod.customer_reports.bad_chan_callsign)
)
GROUP BY ALL;

In [0]:
%sql
WITH time_minute AS (
  SELECT *
  FROM prod.detection.time_minute
  WHERE minute_start >= '2025-10-22 00:00:00'
    AND minute_start <= '2025-10-22 01:00:00'
)
(SELECT tm.minute_start
, CASE WHEN vc.tivo_episode_id IS NULL THEN 'Null Session' Else 'Detected' END AS session_type
, 'Existing Golden' AS table_name
, COUNT(DISTINCT vc.fk_tvid) AS total_tvs
FROM time_minute tm
JOIN prod.detection.viewing_content_golden vc
  ON tm.minute_start >= vc.session_start
  AND tm.minute_start < vc.session_end
  AND vc.session_start_hour >= '2025-10-22 00:00:00'
  AND vc.session_start_hour < '2025-10-22 01:00:00'
GROUP BY 1, 2)
UNION
(SELECT tm.minute_start
, CASE WHEN vc.tivo_episode_id IS NULL THEN 'Null Session' Else 'Detected' END AS session_type
, 'New Golden with Left Join on populations' AS table_name
, COUNT(DISTINCT vc.fk_tvid) AS total_tvs
FROM time_minute tm
JOIN dev.mohit_gangwani.testing_backfill_aws_drop_golden_20251020 vc
  ON tm.minute_start >= vc.session_start
  AND tm.minute_start < vc.session_end
GROUP BY 1, 2)
ORDER BY 1,2,3

Databricks visualization. Run in Databricks to view.

In [0]:

%sql
SELECT tm.minute_start
, CASE WHEN vc.tivo_episode_id IS NULL THEN 'Null Session' Else 'Detected' END AS session_type
-- , CASE WHEN vc.input_category = 'APPS' THEN 'Apps' ELSE 'Other'
-- END AS viewing_type
, COUNT(DISTINCT vc.fk_tvid) AS total_tvs
FROM prod.detection.time_minute tm
JOIN dev.mohit_gangwani.testing_backfill_aws_drop_golden_20251020 vc
  ON tm.minute_start >= vc.session_start
  AND tm.minute_start < vc.session_end
  AND vc.session_start_hour >= '2025-10-20 06:00:00'
  AND vc.session_start_hour < '2025-10-20 07:00:00'
JOIN (
  SELECT tvid, token
  FROM (
    SELECT tvid, token, oem
    , ROW_NUMBER() OVER (PARTITION BY token ORDER BY joined_date DESC) AS rn
    FROM prod.detection.tv
  ) AS tv
  WHERE UPPER(tv.oem) = 'VIZIO'
    AND tv.rn = 1
) AS tv
  ON tv.tvid = vc.fk_tvid
WHERE
  tm.minute_start >= '2025-10-20 06:00:00'
  AND tm.minute_start <= '2025-10-20 07:00:00'
GROUP BY 1, 2

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT tm.minute_start
, CASE WHEN vc.fk_content_id = 3468026 THEN 'Null Session' Else 'Detected' END AS session_type
, COUNT(DISTINCT vc.fk_tvid) AS total_tvs
FROM prod.detection.viewing_content_firehose vc
JOIN (
  SELECT tvid, token
  FROM (
    SELECT tvid, token, oem
    , ROW_NUMBER() OVER (PARTITION BY token ORDER BY joined_date DESC) AS rn
    FROM prod.detection.tv
  ) AS tv
  WHERE UPPER(tv.oem) = 'VIZIO'
    AND tv.rn = 1
) AS tv
  ON tv.tvid = vc.fk_tvid
JOIN prod.detection.time_minute tm
  ON tm.minute_start >= vc.session_start
  AND tm.minute_start < vc.session_end
  AND tm.minute_start >= '2025-10-20 06:00:00'
  AND tm.minute_start <= '2025-10-20 07:00:00'
WHERE vc.session_start >= '2025-10-20 06:00:00'
  AND vc.session_start < '2025-10-20 07:00:00'
  AND vc.fk_zoo_id = 17
  AND vc.session_duration > 0
GROUP BY 1, 2

In [0]:
%sql
WITH viewing_content AS (
  SELECT fk_tvid, session_start, session_end
  FROM prod.detection.viewing_content_firehose vc
  JOIN (
    SELECT tvid, token
    FROM (
      SELECT tvid, token, oem
      , ROW_NUMBER() OVER (PARTITION BY token ORDER BY joined_date DESC) AS rn
      FROM prod.detection.tv
    ) AS tv
    WHERE UPPER(tv.oem) = 'VIZIO'
      AND tv.rn = 1
  ) AS tv
    ON tv.tvid = vc.fk_tvid
  WHERE vc.session_start >= '2025-10-20 06:00:00'
    AND vc.session_start < '2025-10-20 07:00:00'
    AND vc.fk_zoo_id = 17
    AND vc.session_duration > 0
)
, golden AS (
  SELECT fk_tvid, session_start, session_end
  FROM prod.detection.viewing_content_golden vc
  WHERE vc.session_start >= '2025-10-20 06:00:00'
    AND vc.session_start < '2025-10-20 07:00:00'
)
, missing_from_golden AS (
  SELECT content.*
  FROM viewing_content content
  LEFT JOIN golden
    ON golden.fk_tvid = content.fk_tvid
    AND golden.session_start = content.session_start
    AND golden.session_end = content.session_end
  WHERE golden.fk_tvid IS NULL
  GROUP BY ALL
)
SELECT vc.*
FROM prod.detection.viewing_content_firehose vc
JOIN missing_from_golden mg
  ON mg.fk_tvid = vc.fk_tvid
  AND mg.session_start = vc.session_start
  AND mg.session_end = vc.session_end
ORDER BY vc.fk_tvid, vc.session_start
LIMIT 1000

In [0]:
%sql
SELECT DATE_TRUNC('HOUR', vc.session_start) AS session_hour
, vc.fk_content_id IS NULL AS null_content_id
, COUNT(*) AS session_count
FROM prod.detection.viewing_content_firehose vc
JOIN (
  SELECT tvid, token
  FROM (
    SELECT tvid, token, oem
    , ROW_NUMBER() OVER (PARTITION BY token ORDER BY joined_date DESC) AS rn
    FROM prod.detection.tv
  ) AS tv
  WHERE UPPER(tv.oem) = 'VIZIO'
    AND tv.rn = 1
) AS tv
  ON tv.tvid = vc.fk_tvid
WHERE vc.session_start >= '2025-10-20 00:00:00'
  -- AND vc.session_start < '2025-10-21 07:00:00'
  AND vc.fk_zoo_id = 17
  AND vc.session_duration > 0
GROUP BY 1, 2

In [0]:
%sql
SELECT cook.tvid IS NOT NULL, cid.content_cid IS NULL, sch.fk_show_id IS NULL, COUNT(*)
FROM prod.detection.viewing_content_firehose vc
LEFT JOIN prod.cooker.vizio_content_firehose cook
  ON cook.tvid = vc.fk_tvid
 AND cook.ts_start = vc.session_start
 AND cook.ts_end = vc.session_end
LEFT JOIN prod.detection.content_ids_firehose cid
  ON cid.content_cid = cook.cid
LEFT JOIN prod.detection.epg_schedule sch
  ON sch.schedule_id = cid.fk_schedule_id
WHERE vc.fk_content_id IS NULL
  AND vc.session_start >= '2025-10-22 04:00:00'
  AND vc.session_start < '2025-10-22 05:00:00'
GROUP BY 1, 2, 3
-- ORDER BY session_start


In [0]:
%sql
SELECT *

In [0]:
%sql
SELECT vc.created_at AS content_create_ts, cook.created_at AS cooker_create, cid.created_at AS cid_create_ts, COUNT(*)
FROM prod.detection.viewing_content_firehose vc
JOIN prod.cooker.vizio_content_firehose cook
  ON cook.tvid = vc.fk_tvid
 AND cook.ts_start = vc.session_start
 AND cook.ts_end = vc.session_end
JOIN prod.detection.content_ids_firehose cid
  ON cid.content_cid = cook.cid
-- WHERE NOT(vc.fk_content_id <=> 3468026)
--   AND vc.fk_content_id IS NOT NULL
WHERE vc.fk_content_id IS NULL
  AND vc.session_start >= '2025-10-22 04:00:00'
  AND vc.session_start < '2025-10-22 05:00:00'
GROUP BY 1, 2, 3
-- ORDER BY session_start


In [0]:
%sql
SELECT * FROM prod.detection.epg_schedule
WHERE fk_show_id = 1918594
  AND fk_station_id = 92977
  AND airdate = '2025-10-20T06:00:00'

In [0]:
%sql
SELECT * FROM prod.detection.firmware
WHERE firmware_version LIKE '8.%'

In [0]:
%sql
SELECT * FROM prod.cooker.vizio_content_firehose
WHERE tvid = 3486000
AND ts_start = '2025-10-20T06:46:43'

In [0]:
%sql
SELECT * FROM prod.detection.content_ids_firehose
WHERE content_cid = '12108083370_NEWSNTN_2025-10-20T06:00:00Z'

In [0]:
%sql
SELECT * FROM prod.detection.epg_schedule
WHERE schedule_id = 3653801804

In [0]:
%sql
SELECT * FROM system.information_schema.columns
WHERE column_name ILIKE 'device_vendor_id'

In [0]:
%sql
SELECT DATE_TRUNC('HOUR', session_start) AS session_start
, CASE WHEN tivo_episode_id IS NULL THEN 'Null Session' Else 'Detected' END AS session_type
, COUNT(DISTINCT c.fk_tvid||'_'||c.session_start) AS session_count
FROM dev.detection.viewing_content_golden c
WHERE session_start >= '2025-10-20 00:00:00'
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
%sql
SELECT DATE_TRUNC('HOUR', session_start) AS session_start
, CASE WHEN tivo_episode_id IS NULL THEN 'Null Session' Else 'Detected' END AS session_type
, COUNT(DISTINCT c.fk_tvid||'_'||c.session_start) AS session_count
FROM prod.detection.viewing_content_golden c
WHERE session_start >= '2025-10-20 00:00:00'
GROUP BY 1, 2

In [0]:
%sql
SELECT DATE_TRUNC('HOUR', session_start) AS session_start
, CASE WHEN c.fk_content_id <=> 3468026 THEN 'Null Session' Else 'Detected' END AS session_type
, COUNT(DISTINCT c.fk_tvid||'_'||c.session_start) AS session_count
FROM prod.detection.viewing_content_firehose c
-- JOIN prod.detection.tv AS tv
--   ON c.fk_tvid = tv.tvid
--  AND tv.oem = 'VIZIO'
-- JOIN prod.detection.tv_settings AS tv_settings
--   ON c.session_start >= tv_settings.create_timestamp
--  AND c.session_start < tv_settings.next_create_timestamp
--  AND tv_settings.next_create_timestamp >= '2025-10-20 00:00:00'::timestamp
--  AND c.fk_tvid = tv_settings.fk_tvid
-- JOIN prod.detection.settings AS settings
--   ON tv_settings.fk_settings_id = settings.settings_id
--  AND UPPER(settings.country_name) = 'USA'
-- JOIN prod.detection.tv_populations AS u
--   ON c.fk_tvid = u.fk_tvid
-- JOIN prod.detection.populations AS pop
--   ON u.fk_population_id = pop.population_id
--  AND pop.population_name = 'opted_in'
-- JOIN prod.detection.location AS location
--   ON c.fk_location_id = location.location_id
--  AND UPPER(location.country_code) = 'US'
-- JOIN prod.detection.tv_input_stats_firehose AS tvis
--   ON c.session_start >= tvis.create_timestamp
--  AND c.session_start < tvis.next_create_timestamp
--  AND tvis.next_create_timestamp >= '2025-10-20 00:00:00'::timestamp
--  AND c.fk_tvid = tvis.fk_tvid
--  AND c.fk_input_source_id = tvis.fk_input_source_id
WHERE c.session_start >= '2025-10-20 00:00:00'
  AND c.fk_zoo_id = 17
GROUP BY 1, 2
ORDER BY 1, 2